# CUM Series 11 — Learned Polynomial + Per-Layer Adaptive + Selective Un-Equalization

**Three new directions:**
1. **11v1**: Custom NS polynomial targeting stable SVs=0.88 (no oscillation)
2. **11v2**: Per-layer adaptive blend weights based on change ratio
3. **11v3**: Light (5%) row-wise curvature recovery after combined mode

**Baseline:** Combined mode (iterate+input blend) = -0.019 vs Muon in best run

In [ ]:
# ── Setup: clone repo, verify GPU, enable CUDA optimizations ──
import os, subprocess, sys

REPO_URL = 'https://github.com/BrownHujay/Curvature-Unified-Muon.git'
REPO_DIR = '/content/Curvature-Unified-Muon'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print(f'Cloned repo')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print(f'Pulled latest')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math
import numpy as np

# CUDA optimizations
torch.set_float32_matmul_precision('high')  # enable TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

assert torch.cuda.is_available(), 'Need GPU!'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'TF32 enabled: {torch.backends.cuda.matmul.allow_tf32}')

In [ ]:
# ── Download data ──
DATA_DIR = 'benchmarks/tier0/data'
DATA_PATH = os.path.join(DATA_DIR, 'tinyshakespeare.txt')
if not os.path.exists(DATA_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    import urllib.request
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
        DATA_PATH)
with open(DATA_PATH, 'r') as f:
    TEXT = f.read()
print(f'Dataset: {len(TEXT):,} chars')

In [ ]:
# ── Imports ──
from benchmarks.models.micro_gpt import MicroGPT
from cum.newton_schulz import newton_schulz_orthogonalize
from cum.utils import aspect_ratio_scale
from cum import CUM5v6, CUM8v1, CUM9v1, CUM11v1, CUM11v2, CUM11v3
print('Imports OK')

In [ ]:
# ── Config (proven: no overfitting on TinyShakespeare) ──
MODEL_CFG = dict(
    d_model=128,
    n_heads=4,
    n_layers=4,
    d_ff=512,
    ctx_len=256,
)
BATCH_SIZE = 32
TOTAL_STEPS = 2000
WARMUP_STEPS = 200
EVAL_EVERY = 250
BASE_LR = 0.02
SEED = 42

# Sanity check
test_model = MicroGPT(vocab_size=65, **MODEL_CFG)
n_params = sum(p.numel() for p in test_model.parameters())
n_hidden = sum(p.numel() for p in test_model.get_hidden_2d_params())
hidden_shapes = [(p.shape[0], p.shape[1]) for p in test_model.get_hidden_2d_params()]
print(f'Model: {n_params/1e6:.1f}M params ({n_hidden/1e6:.1f}M hidden 2D)')
print(f'Hidden matrix shapes: {set(hidden_shapes)}')
print(f'Batch={BATCH_SIZE}, Steps={TOTAL_STEPS}')
del test_model

In [ ]:
# ── Data & Training Infrastructure ──

class CharDataset:
    def __init__(self, text, ctx_len=256, device='cpu'):
        self.chars = sorted(set(text))
        self.char_to_idx = {c: i for i, c in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
        self.data = torch.tensor([self.char_to_idx[c] for c in text], dtype=torch.long, device=device)
        self.ctx_len = ctx_len

    def get_batch(self, batch_size, split='train', train_ratio=0.9):
        n = int(len(self.data) * train_ratio)
        data = self.data[:n] if split == 'train' else self.data[n:]
        ix = torch.randint(0, len(data) - self.ctx_len - 1, (batch_size,), device=self.data.device)
        x = torch.stack([data[i:i+self.ctx_len] for i in ix])
        y = torch.stack([data[i+1:i+self.ctx_len+1] for i in ix])
        return x, y


class SimpleMuon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, beta1=0.95, ns_steps=5):
        defaults = dict(lr=lr, beta1=beta1, ns_steps=ns_steps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                state = self.state[p]
                if len(state) == 0:
                    state['momentum_buffer'] = torch.zeros_like(p)
                m = state['momentum_buffer']
                m.mul_(group['beta1']).add_(p.grad, alpha=1 - group['beta1'])
                u = p.grad + group['beta1'] * m
                orth = newton_schulz_orthogonalize(u, steps=group['ns_steps'])
                scale = math.sqrt(max(1, p.shape[0] / p.shape[1]))
                p.add_(orth, alpha=-group['lr'] * scale)


def get_lr(step, total_steps, warmup_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


dataset = CharDataset(TEXT, ctx_len=MODEL_CFG['ctx_len'], device=DEVICE)
print(f'Dataset on {DEVICE}: vocab={dataset.vocab_size}')

In [ ]:
# ── Training Loop ──

def train_one(name, main_opt, aux_opt, model, dataset,
              total_steps=TOTAL_STEPS, batch_size=BATCH_SIZE,
              warmup_steps=WARMUP_STEPS, base_lr=BASE_LR, eval_every=EVAL_EVERY):
    model.train()
    trajectory = []

    # Warmup torch.compile
    for _ in range(3):
        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)
        loss.backward()
        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    for step in range(total_steps):
        current_lr = get_lr(step, total_steps, warmup_steps, base_lr)
        if main_opt:
            for g in main_opt.param_groups:
                g['lr'] = current_lr

        x, y = dataset.get_batch(batch_size, split='train')
        _, loss = model(x, y)

        if main_opt: main_opt.zero_grad()
        aux_opt.zero_grad()
        loss.backward()
        if main_opt: main_opt.step()
        aux_opt.step()

        if step % eval_every == 0 or step == total_steps - 1:
            model.eval()
            vl = []
            for _ in range(20):
                vx, vy = dataset.get_batch(batch_size, split='val')
                with torch.no_grad():
                    _, v = model(vx, vy)
                    vl.append(v.item())
            val_loss = np.mean(vl)
            trajectory.append((step, val_loss))
            model.train()
            elapsed = time.perf_counter() - t0
            print(f'  [{name}] step {step}: val_loss={val_loss:.4f} ({elapsed:.0f}s)')

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    # Final eval (more samples)
    model.eval()
    vl = []
    for _ in range(50):
        vx, vy = dataset.get_batch(batch_size, split='val')
        with torch.no_grad():
            _, v = model(vx, vy)
            vl.append(v.item())
    final_val = np.mean(vl)
    return final_val, trajectory, elapsed


print('Training loop ready')

In [ ]:
# ── Optimizer Factory ──

def make_model_and_opts(dataset, cfg):
    torch.manual_seed(SEED)
    model = MicroGPT(vocab_size=dataset.vocab_size, **MODEL_CFG).to(DEVICE)
    model = torch.compile(model, mode='reduce-overhead')

    hidden_2d = model.get_hidden_2d_params()
    other = model.get_other_params()
    aux_opt = torch.optim.AdamW(other, lr=3e-4, weight_decay=0.01)

    t = cfg['type']

    if t == 'Muon':
        main_opt = SimpleMuon(hidden_2d, lr=BASE_LR, ns_steps=5)
    elif t == '5v6':
        main_opt = CUM5v6(
            hidden_2d, lr=BASE_LR,
            mode=cfg.get('mode', 'ns_blend'),
            ns_save_at=cfg.get('ns_save_at', 2),
            ns_blend=cfg.get('ns_blend', 0.25),
        )
    elif t == '8v1':
        main_opt = CUM8v1(
            hidden_2d, lr=BASE_LR,
            method=cfg.get('method', 'matrix'),
            mode=cfg.get('mode', 'two_point'),
            ns_steps=cfg.get('ns_steps', 5),
            save_at=cfg.get('save_at', 2),
            blend=cfg.get('blend', 0.15),
            save_at_a=cfg.get('save_at_a', 1),
            save_at_b=cfg.get('save_at_b', 3),
            blend_a=cfg.get('blend_a', 0.05),
            blend_b=cfg.get('blend_b', 0.10),
            sv_blend_mode=cfg.get('sv_blend_mode', 'arithmetic'),
            input_blend_beta=cfg.get('input_blend_beta', 0.5),
            input_blend_alpha=cfg.get('input_blend_alpha', 0.15),
            alpha_curv=cfg.get('alpha_curv', 0.15),
            beta_temp=cfg.get('beta_temp', 0.15),
            total_steps=TOTAL_STEPS,
        )
    elif t == '9v1':
        main_opt = CUM9v1(
            hidden_2d, lr=BASE_LR,
            beta_fast=cfg.get('beta_fast', 0.80),
            beta_slow=cfg.get('beta_slow', 0.95),
            blend=cfg.get('blend', 0.15),
            mode=cfg.get('mode', 'pre_ns'),
            ns_steps=5,
        )
    elif t == '11v1':
        main_opt = CUM11v1(
            hidden_2d, lr=BASE_LR,
            ns_a=cfg.get('ns_a', 3.4445),
            ns_b=cfg.get('ns_b', -4.7750),
            ns_c=cfg.get('ns_c', 2.0315),
            mode=cfg.get('mode', 'basic'),
            ns_steps=cfg.get('ns_steps', 5),
            save_at=cfg.get('save_at', 2),
            blend=cfg.get('blend', 0.15),
            input_blend_beta=cfg.get('input_blend_beta', 0.5),
            input_blend_alpha=cfg.get('input_blend_alpha', 0.15),
        )
    elif t == '11v2':
        main_opt = CUM11v2(
            hidden_2d, lr=BASE_LR,
            ns_steps=cfg.get('ns_steps', 5),
            save_at=cfg.get('save_at', 2),
            base_blend=cfg.get('base_blend', 0.15),
            base_input_alpha=cfg.get('base_input_alpha', 0.15),
            adapt_scale=cfg.get('adapt_scale', 1.0),
            input_blend_beta=cfg.get('input_blend_beta', 0.5),
        )
    elif t == '11v3':
        main_opt = CUM11v3(
            hidden_2d, lr=BASE_LR,
            ns_steps=cfg.get('ns_steps', 5),
            save_at=cfg.get('save_at', 2),
            blend=cfg.get('blend', 0.15),
            input_blend_beta=cfg.get('input_blend_beta', 0.5),
            input_blend_alpha=cfg.get('input_blend_alpha', 0.15),
            uneq_alpha=cfg.get('uneq_alpha', 0.05),
            uneq_beta=cfg.get('uneq_beta', 0.99),
        )
    else:
        raise ValueError(f'Unknown: {t}')

    return model, main_opt, aux_opt


def run_all(configs):
    results = []
    for i, cfg in enumerate(configs):
        name = cfg['name']
        print(f'\n{"="*60}')
        print(f'[{i+1}/{len(configs)}] {name}')
        print(f'{"="*60}')
        try:
            model, main_opt, aux_opt = make_model_and_opts(dataset, cfg)
            val_loss, traj, elapsed = train_one(name, main_opt, aux_opt, model, dataset)
            results.append(dict(name=name, type=cfg['type'], val_loss=val_loss,
                                trajectory=traj, elapsed=elapsed, error=None))
            print(f'  FINAL: {name} -> {val_loss:.4f} ({elapsed:.1f}s)')
        except Exception as e:
            import traceback; traceback.print_exc()
            results.append(dict(name=name, type=cfg['type'], val_loss=float('inf'),
                                trajectory=[], elapsed=0, error=str(e)))
        torch.cuda.empty_cache()
    return results


def show_results(results, series='11a'):
    muon = next((r['val_loss'] for r in results if r['type'] == 'Muon'), None)
    combined_ref = next((r['val_loss'] for r in results
                         if r.get('name', '').startswith('8v1 combined')), None)

    print(f'\n## Series {series} Results\n')
    print(f'| Config | Val Loss | vs Muon | vs Combined | Time |')
    print(f'|--------|----------|---------|-------------|------|')
    for r in sorted(results, key=lambda x: x['val_loss']):
        if r['error']:
            print(f'| {r["name"]} | FAILED | — | — | {r["error"][:30]} |')
            continue
        vm = f'{r["val_loss"] - muon:+.4f}' if muon else '—'
        vc = f'{r["val_loss"] - combined_ref:+.4f}' if combined_ref else '—'
        print(f'| {r["name"]} | {r["val_loss"]:.4f} | {vm} | {vc} | {r["elapsed"]:.0f}s |')

    new = [r for r in sorted(results, key=lambda x: x['val_loss'])
           if r['type'] not in ('Muon',) and not r['error']
           and not r.get('name', '').startswith('8v1 combined')]
    if new:
        b = new[0]
        print(f'\nBEST NEW: {b["name"]} -> {b["val_loss"]:.4f}', end='')
        if muon: print(f' (vs Muon: {b["val_loss"]-muon:+.4f})', end='')
        if combined_ref: print(f' (vs Combined: {b["val_loss"]-combined_ref:+.4f})', end='')
        print()


print('Factory ready')

---
## Series 11a: Learned Polynomial Coefficients

**Q1:** Does a polynomial with a STABLE fixed point at 0.88 (no oscillation) beat Muon?

**Q2:** Does stable polynomial + combined mode (double cancellation) beat standard combined?

Stable-0.88 polynomial: `(a, b, c) = (2.0, -1.940, 0.836)`
- `p(0.88) ≈ 0.88` (IS a fixed point)
- `p'(0.88) ≈ 0.0` (super-stable, derivative near zero)
- Small SVs grow: `0.1 → 0.198 → 0.385 → 0.640 → 0.819 → 0.873` in 5 steps

In [ ]:
CONFIGS_11A = [
    {'name': 'Muon NS=5',              'type': 'Muon'},
    # Replicate combined mode (standard poly + double cancellation)
    {'name': '8v1 combined (std poly)',  'type': '8v1', 'method': 'matrix', 'mode': 'combined',
     'save_at': 2, 'blend': 0.15, 'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
    # Stable polynomial alone (no blending — does it need cancellation?)
    {'name': '11v1 stable basic',       'type': '11v1',
     'ns_a': 2.0, 'ns_b': -1.940, 'ns_c': 0.836, 'mode': 'basic'},
    # Stable polynomial + double cancellation
    {'name': '11v1 stable combined',    'type': '11v1',
     'ns_a': 2.0, 'ns_b': -1.940, 'ns_c': 0.836, 'mode': 'combined',
     'save_at': 2, 'blend': 0.15, 'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
]

print(f'{len(CONFIGS_11A)} configs')

In [ ]:
results_11a = run_all(CONFIGS_11A)
show_results(results_11a, '11a')

---
## Series 11b: Per-Layer Adaptive + Selective Un-Equalization

**Q3:** Does adapting blend weight per-layer (by how much NS changes each layer) help?

**Q4:** Does restoring 5% of curvature (row-wise) after combined mode help?

In [ ]:
CONFIGS_11B = [
    {'name': 'Muon NS=5',              'type': 'Muon'},
    # Combined reference
    {'name': '8v1 combined (ref)',       'type': '8v1', 'method': 'matrix', 'mode': 'combined',
     'save_at': 2, 'blend': 0.15, 'input_blend_beta': 0.5, 'input_blend_alpha': 0.15},
    # Per-layer adaptive: scale=1.0 means change_ratio doubles the blend if NS fully restructures
    {'name': '11v2 adaptive s=1.0',     'type': '11v2', 'adapt_scale': 1.0},
    # Selective un-equalization: 5% curvature recovery
    {'name': '11v3 uneq a=0.05',       'type': '11v3', 'uneq_alpha': 0.05},
]

print(f'{len(CONFIGS_11B)} configs')

In [ ]:
results_11b = run_all(CONFIGS_11B)
show_results(results_11b, '11b')

---
## Series 11c: Follow-up (fill based on 11a/11b results)

In [ ]:
# TODO: fill in based on 11a/11b results
# Ideas:
# - If stable basic beats Muon: try stable + input_blend (cheaper than combined)
# - If stable combined wins: try different coefficient triples
# - If adaptive wins: sweep adapt_scale (0.5, 2.0)
# - If uneq wins: sweep uneq_alpha (0.02, 0.10)
# - Best from 11a + best from 11b combined

CONFIGS_11C = [
    {'name': 'Muon NS=5',              'type': 'Muon'},
    # Add refinements here
]

if len(CONFIGS_11C) > 1:
    results_11c = run_all(CONFIGS_11C)
    show_results(results_11c, '11c')
else:
    print('Fill in configs based on 11a/11b results, then re-run')